# My Validation and Research Claim Audit

I want to check how much trust the Week 5 result deserves. I start with two methodology questions about the supplied paper, then compare row-random and client-held-out validation, broaden the client check, audit leakage, and rewrite the claims.

This is still a March development exercise. I am not opening the final month or calling the result a production forecast.

## 1. Two paper findings and my methodology questions

Source: [FlyRank — The State of AI-Driven SEO, March 2026](https://github.com/flyrank-bih/flyrank-ml-internship-starter/blob/main/docs/flyrank-seo-research-march-2026.pdf). I read the modelling appendix and methodology, not just the headline scores.

**Finding 1, page 27:** The random forest assigns 43% importance to average position when predicting health score. The paper discloses that the composite target partly contains position and other inputs; it describes the importance as descriptive. My question is: how much importance remains when evaluating an independent later search outcome, with clients or periods held aside? That would extend the disclosed result before turning it into an optimisation priority.

**Finding 2, page 29:** The growth/decline logistic regression reports 71% holdout accuracy. Page 36 describes an 80/20 split of the active-content sample; page 5 defines the trend buckets using a 30-day comparison and ±10% thresholds. I would ask for the class base rate, whether clients share both sides, and how input dates align with that trend label. A grouped and later-period check would help assess transfer beyond this exploratory sample.

The paper already separates exploratory modelling from its main observational findings. My questions build on those limits; I am not treating its labels, thresholds, or sample as identical to my 14-day, 20%-decline contract.

In [1]:
from pathlib import Path
import importlib.util
import subprocess
import sys

needed = {"duckdb": "duckdb", "huggingface_hub": "huggingface_hub", "sklearn": "scikit-learn",
          "pyarrow": "pyarrow", "matplotlib": "matplotlib", "pandas": "pandas"}
missing = [package for module, package in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "work/lane_utils.py").is_file()), None)
if root is None:
    if importlib.util.find_spec("google.colab") is None:
        raise RuntimeError("Open this notebook inside your repository checkout.")
    root = Path("/content/FlyrankAI-Internship")
    if not root.exists():
        subprocess.check_call(["git", "clone", "--depth", "1",
            "https://github.com/vibhanshu-s/FlyrankAI-Internship.git", str(root)])
sys.path.insert(0, str(root / "work"))
import lane_utils as u
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pages = u.load_pages(root)
experiment = u.labelled_pages(pages)
train_index, test_index = u.grouped_split(experiment)
training = experiment.iloc[train_index].copy()
test = experiment.iloc[test_index].copy()
print(f"March review pool: {len(pages):,} pages; known outcomes: {len(experiment):,}; unknown: {pages.future_decline.isna().sum():,}.")
print(f"Fixed client split: {len(training):,} training pages / {len(test):,} test pages.")
print("Features stop March 14; assumed decision March 18; labels use March 18–31. June remains sealed.")

March review pool: 60,534 pages; known outcomes: 57,459; unknown: 3,075.
Fixed client split: 14,070 training pages / 43,389 test pages.
Features stop March 14; assumed decision March 18; labels use March 18–31. June remains sealed.


## 2. My model under an honest split: before and after

Week 5 already used a client-held-out split. Here I deliberately show a weaker **row-random split** beside the same **client-held-out split**, with the model and rule fixed. Moving from shared clients to zero client overlap improves the validation design, even if the score does not move in the direction I expected.

The row-random test contains only a subset of each client's pages, while the client-held-out test contains whole held-out inventories. Their populations and base rates differ, so a score gap is not a controlled measure of memorisation. I compare model versus rule within each split, not select whichever split produces a nicer result.

I then improve coverage beyond Week 5's single split using **five-fold GroupKFold**. Each client's pages receive scores from a model trained on other clients. Every labelled page is evaluated once. The final comparison ranks these out-of-fold scores within clients, using the same review limit.

In [2]:
import json
from sklearn.model_selection import train_test_split, GroupKFold
model_receipt = json.loads((root / "work/outputs/w05_model_metrics.json").read_text(encoding="utf-8"))
assert model_receipt["data_fingerprint"] == u.fingerprint(pages)
assert model_receipt["split"] == u.split_fingerprint(experiment, train_index, test_index)
assert model_receipt["baseline"] == u.BASELINE
X, y = experiment[u.FEATURES], experiment.future_decline.astype(int)
row_train, row_test = train_test_split(np.arange(len(experiment)), test_size=0.25, random_state=42, stratify=y)
split_results = {}
overlap_counts = {}
for name, train_rows, test_rows in [
    ("Row-random (weaker)", row_train, row_test),
    ("Client-held-out (Week 5)", train_index, test_index),
]:
    fitted = u.make_model()
    fitted.fit(X.iloc[train_rows], y.iloc[train_rows])
    predicted = fitted.predict_proba(X.iloc[test_rows])[:, 1]
    subset = experiment.iloc[test_rows]
    overlap = set(experiment.iloc[train_rows].client_hash_id) & set(subset.client_hash_id)
    overlap_counts[name] = len(overlap)
    for method, scores in [("Frozen rule", u.rule_score(subset)), ("Logistic regression", predicted)]:
        split_results[f"{name} / {method}"] = u.evaluate(subset, scores)
    if name.startswith("Client"):
        grouped_probability = predicted
        grouped_model = fitted
        assert not overlap
        assert np.isclose(split_results[f"{name} / Logistic regression"]["macro_precision_at_20"],
                          model_receipt["metrics"]["Logistic regression"]["macro_precision_at_20"])
display(u.summary_table(split_results))
display(pd.DataFrame({"shared_clients_between_train_and_test": overlap_counts}))
print("Before/after improvement: shared-client rows are replaced by whole-client holdout.")
print("Different test inventories and base rates prevent treating the gap as a pure memorisation effect.")

,macro_precision_at_20,macro_base_rate_same_clients,roc_auc,average_precision,full_queue_clients,short_queue_clients,pages
Row-random (weaker) / Frozen rule,0.3550,0.3833,0.5118,0.3642,20.0,9.0,14365.0
Row-random (weaker) / Logistic regression,0.5250,0.3833,0.6194,0.4767,20.0,9.0,14365.0
Client-held-out (Week 5) / Frozen rule,0.3714,0.3981,0.5160,0.3811,7.0,1.0,43389.0
Client-held-out (Week 5) / Logistic regression,0.5714,0.3981,0.6216,0.4876,7.0,1.0,43389.0


,shared_clients_between_train_and_test
Row-random (weaker),29
Client-held-out (Week 5),0


Before/after improvement: shared-client rows are replaced by whole-client holdout.
Different test inventories and base rates prevent treating the gap as a pure memorisation effect.


In [3]:
oof = np.full(len(experiment), np.nan)
fold_id = np.zeros(len(experiment), dtype=int)
fold_rows = []
for fold, (train_rows, test_rows) in enumerate(GroupKFold(n_splits=5).split(X, y, experiment.client_hash_id), 1):
    assert set(experiment.iloc[train_rows].client_hash_id).isdisjoint(set(experiment.iloc[test_rows].client_hash_id))
    assert np.isnan(oof[test_rows]).all()
    fitted = u.make_model()
    fitted.fit(X.iloc[train_rows], y.iloc[train_rows])
    oof[test_rows] = fitted.predict_proba(X.iloc[test_rows])[:, 1]
    fold_id[test_rows] = fold
    subset = experiment.iloc[test_rows]
    for method, scores in [("Frozen rule", u.rule_score(subset)), ("Logistic regression", oof[test_rows])]:
        metrics = u.evaluate(subset, scores)
        fold_rows.append({"fold": fold, "method": method, **metrics})
assert np.isfinite(oof).all() and (fold_id > 0).all()
fold_table = pd.DataFrame(fold_rows)
display(fold_table[["fold", "method", "pages", "clients", "macro_precision_at_20", "macro_base_rate_same_clients", "roc_auc"]].round(4))
oof_results = {"Frozen rule / all OOF pages": u.evaluate(experiment, u.rule_score(experiment)),
               "Logistic regression / all OOF pages": u.evaluate(experiment, oof)}
display(u.summary_table(oof_results))

# Estimate uncertainty by resampling paired client results, not individual pages.
rule_rank = u.rank_frame(experiment, u.rule_score(experiment))
model_rank = u.rank_frame(experiment, oof)
rule_per_client = rule_rank.loc[rule_rank.client_rank.le(20)].groupby("client_hash_id").future_decline.agg(["mean", "size"])
model_per_client = model_rank.loc[model_rank.client_rank.le(20)].groupby("client_hash_id").future_decline.agg(["mean", "size"])
full_clients = rule_per_client.index[rule_per_client["size"].eq(20)]
assert model_per_client.loc[full_clients, "size"].eq(20).all()
client_delta = (model_per_client.loc[full_clients, "mean"] - rule_per_client.loc[full_clients, "mean"]).to_numpy(dtype=float)
rng = np.random.default_rng(42)
bootstrap_means = rng.choice(client_delta, size=(2000, len(client_delta)), replace=True).mean(axis=1)
lower, upper = np.quantile(bootstrap_means, [0.025, 0.975])
delta = float(client_delta.mean())
fold_gain = []
for fold in range(1, 6):
    pair = fold_table.loc[fold_table.fold.eq(fold)].set_index("method")
    fold_gain.append(float(pair.loc["Logistic regression", "macro_precision_at_20"] - pair.loc["Frozen rule", "macro_precision_at_20"]))
print(f"All-OOF mean precision@20 gain: {100 * delta:+.2f} percentage points across {len(full_clients)} full-list clients.")
print(f"Paired-client bootstrap 95% interval: [{100 * lower:+.2f}, {100 * upper:+.2f}] points.")
print(f"Positive model-minus-rule gains in {sum(value > 0 for value in fold_gain)} / 5 folds.")
print("The interval resamples clients conditional on fitted fold models; it does not include refitting or month-to-month uncertainty.")

oof_frame = experiment.copy()
oof_frame["oof_score"] = oof
oof_frame["fold"] = fold_id
oof_frame.to_parquet(root / "work/outputs/w06_oof_predictions.parquet", index=False)

,fold,method,pages,clients,macro_precision_at_20,macro_base_rate_same_clients,roc_auc
0,1,Frozen rule,15502,1,0.1000,0.3809,0.5301
1,1,Logistic regression,15502,1,0.6000,0.3809,0.6030
2,2,Frozen rule,12083,1,0.5500,0.4060,0.5514
3,2,Logistic regression,12083,1,0.5500,0.4060,0.6289
4,3,Frozen rule,9958,7,0.5125,0.4886,0.4675
5,3,Logistic regression,9958,7,0.5875,0.4886,0.5400
6,4,Frozen rule,9957,12,0.2875,0.3576,0.5189
7,4,Logistic regression,9957,12,0.5375,0.3576,0.5702
8,5,Frozen rule,9959,11,0.4222,0.4055,0.4809
9,5,Logistic regression,9959,11,0.5944,0.4055,0.6241


,macro_precision_at_20,macro_base_rate_same_clients,roc_auc,average_precision,full_queue_clients,short_queue_clients,pages
Frozen rule / all OOF pages,0.3826,0.4023,0.5170,0.3685,23.0,9.0,57459.0
Logistic regression / all OOF pages,0.5717,0.4023,0.5847,0.4397,23.0,9.0,57459.0


All-OOF mean precision@20 gain: +18.91 percentage points across 23 full-list clients.
Paired-client bootstrap 95% interval: [+11.52, +26.74] points.
Positive model-minus-rule gains in 4 / 5 folds.
The interval resamples clients conditional on fitted fold models; it does not include refitting or month-to-month uncertainty.


## 3. Leakage audit and real failures

Every feature comes from March 1–14. The target and its later-window inputs stay outside the feature frame; IDs only group or identify pages. Scaling and median imputation fit inside each training split. The frozen rule is a comparison, never a model feature. I do not use product scores, trend buckets, or the fixed 90-day query table.

Two limits remain important. First, the evaluation population requires complete **later** coverage. That is a retrospective selection condition and can exclude pages that lost tracking; it cannot be used to claim coverage of all review candidates. Second, the models learn from other clients' late-March labels. They were not actually trained and available on March 18. This is an offline cross-client exercise, not a faithful historical deployment test. A real forecast needs an earlier completed training period and a later untouched outcome period.

The earlier impression total is also the denominator of the decline comparison. A temporary earlier spike can create apparent decline through regression to the mean. That is legitimate earlier information, but it complicates any interpretation as content decay.

I repeat the deliberate-leak check as a positive control, then remove the answer column and model. I retain the five earlier-window inputs. Active search days has no variation in this filtered slice, so it cannot be credited with useful signal.

In [4]:
audit_rows = [
    {"feature": name, "window_end": "2026-03-14", "available_before_decision": True,
     "label_or_product_derived": False, "distinct_values": int(experiment[name].nunique(dropna=True))}
    for name in u.FEATURES
]
display(pd.DataFrame(audit_rows))
assert list(X.columns) == u.FEATURES
forbidden = {"future_decline", "later_impressions", "later_days", "leaked_future_decline",
             "trend_pct", "trend_direction", "health_score", "priority_score", *u.KEYS}
assert set(X.columns).isdisjoint(forbidden)
assert pd.Timestamp("2026-03-14") < pd.Timestamp("2026-03-18")
assert np.isfinite(X.to_numpy(dtype=float)).all()
print(f"Later-coverage exclusions: {pages.future_decline.isna().sum():,} pages; retained for the separate review pool.")

from sklearn.metrics import roc_auc_score
leaked_X = X.assign(leaked_future_decline=y.to_numpy())
leaked_model = u.make_model()
leaked_model.fit(leaked_X.iloc[train_index], y.iloc[train_index])
leaked_auc = roc_auc_score(y.iloc[test_index], leaked_model.predict_proba(leaked_X.iloc[test_index])[:, 1])
clean_auc = roc_auc_score(y.iloc[test_index], grouped_probability)
display(pd.DataFrame({"version": ["Clean Week 5 features", "One deliberate copy of the label"],
                      "ROC_AUC": [clean_auc, leaked_auc]}).round(4))
leaked_X.drop(columns=["leaked_future_decline"], inplace=True)
assert list(leaked_X.columns) == u.FEATURES
del leaked_X, leaked_model
print("Leaked input and model removed. All OOF scores used clean features only.")

selected = model_rank.client_rank.le(20)
failure_cases = pd.concat([
    model_rank.loc[selected & model_rank.future_decline.eq(0)].sort_values("score", ascending=False).head(3).assign(error="Selected, no decline"),
    model_rank.loc[~selected & model_rank.future_decline.eq(1)].sort_values("earlier_impressions", ascending=False).head(3).assign(error="Declined, not selected"),
])
display(failure_cases[u.KEYS + ["error", "client_rank", "score", "earlier_impressions",
                              "later_impressions", "ctr_pct", "weighted_position"]])
display(Markdown((chr(10) * 2).join(
    f"- **{row.content_hash_id}:** {row.error}; impressions were {row.earlier_impressions:,.0f} before "
    f"and {row.later_impressions:,.0f} later. This may reflect unmeasured query or timing changes; "
    "the measurements do not establish the cause or a suitable edit."
    for _, row in failure_cases.iterrows())))

,feature,window_end,available_before_decision,label_or_product_derived,distinct_values
0,log_impressions,2026-03-14,True,False,8318
1,log_clicks,2026-03-14,True,False,223
2,ctr_pct,2026-03-14,True,False,16224
3,weighted_position,2026-03-14,True,False,55457
4,active_search_days,2026-03-14,True,False,1


Later-coverage exclusions: 3,075 pages; retained for the separate review pool.


,version,ROC_AUC
0,Clean Week 5 features,0.6216
1,One deliberate copy of the label,1.0000


Leaked input and model removed. All OOF scores used clean features only.


,client_hash_id,content_hash_id,error,client_rank,score,earlier_impressions,later_impressions,ctr_pct,weighted_position
13732,client_23a62021009f63c4,content_b875a2f306635a58,"Selected, no decline",1,0.824755,30998.0,29878.0,0.000000,32.499097
39531,client_73cda7b4e4f265ea,content_8e1334d6356668e3,"Selected, no decline",2,0.785960,56091.0,72636.0,0.001783,4.642973
10591,client_23a62021009f63c4,content_67a19b4e8f52924e,"Selected, no decline",5,0.776455,18119.0,15795.0,0.000000,44.433799
15538,client_23a62021009f63c4,content_e8a52cf3d5988c07,"Declined, not selected",9632,0.115434,138793.0,86531.0,0.244969,16.161262
8770,client_23a62021009f63c4,content_36e53e9c707674fc,"Declined, not selected",8823,0.191182,105148.0,73675.0,0.104614,33.144111
24361,client_62f4a7e64f5e0096,content_7c6373141eae744a,"Declined, not selected",1570,0.435605,84160.0,41458.0,0.055846,5.990078


- **content_b875a2f306635a58:** Selected, no decline; impressions were 30,998 before and 29,878 later. This may reflect unmeasured query or timing changes; the measurements do not establish the cause or a suitable edit.

- **content_8e1334d6356668e3:** Selected, no decline; impressions were 56,091 before and 72,636 later. This may reflect unmeasured query or timing changes; the measurements do not establish the cause or a suitable edit.

- **content_67a19b4e8f52924e:** Selected, no decline; impressions were 18,119 before and 15,795 later. This may reflect unmeasured query or timing changes; the measurements do not establish the cause or a suitable edit.

- **content_e8a52cf3d5988c07:** Declined, not selected; impressions were 138,793 before and 86,531 later. This may reflect unmeasured query or timing changes; the measurements do not establish the cause or a suitable edit.

- **content_36e53e9c707674fc:** Declined, not selected; impressions were 105,148 before and 73,675 later. This may reflect unmeasured query or timing changes; the measurements do not establish the cause or a suitable edit.

- **content_7c6373141eae744a:** Declined, not selected; impressions were 84,160 before and 41,458 later. This may reflect unmeasured query or timing changes; the measurements do not establish the cause or a suitable edit.

## 4. Claim rewrite

**Too strong:** “The model finds pages that need a refresh and will prevent traffic loss.”

**Safer:** “In this March slice, we measured how well an earlier-signal score prioritised a later impression-decline proxy. It may support an editor's review, but neither decline nor the score proves that a refresh is needed or would prevent traffic loss.”

**Too strong:** “The Week 5 improvement shows that the model is ready for any client.”

**Safer:** “The single client-held-out comparison showed a measured improvement over my rule. The broader grouped check below tests whether that improvement travels across more clients in this same month. It does not establish performance in a new month, confidence for every client, or causal value.”

The decision for the playbook uses the audit, not the nicer single-split score. I require at least a five-point mean gain, a positive lower endpoint in the paired-client interval, and positive gains in at least four of five grouped folds to justify a model-led research queue. Otherwise I keep the fixed rule. These are conservative research gates, not production acceptance criteria.

In [5]:
promote_model = bool(delta >= 0.05 and lower > 0 and sum(value > 0 for value in fold_gain) >= 4)
selected_method = "logistic_regression_oof" if promote_model else "frozen_rule"
display(Markdown(
    f"**Measured result:** OOF precision@20 is {100 * oof_results['Logistic regression / all OOF pages']['macro_precision_at_20']:.2f}% "
    f"for logistic regression and {100 * oof_results['Frozen rule / all OOF pages']['macro_precision_at_20']:.2f}% for the rule. "
    f"The paired-client gain interval is [{100 * lower:+.2f}, {100 * upper:+.2f}] points. "
    f"My playbook method is **{selected_method}**. This is a research choice for human review, not an automated editing system."))

receipt = {"contract": u.CONTRACT, "data_fingerprint": u.fingerprint(pages), "baseline": u.BASELINE,
    "week5_split": u.split_fingerprint(experiment, train_index, test_index),
    "paper": {"url": "https://github.com/flyrank-bih/flyrank-ml-internship-starter/blob/main/docs/flyrank-seo-research-march-2026.pdf",
              "sha256": "4c1e59c6e1196af37ed3145b0b7fa964a352dcb920588203a434600483321f82", "finding_pages": [27, 29], "methodology_page": 36},
    "before_after_metrics": split_results, "shared_client_counts": overlap_counts,
    "oof_metrics": oof_results,
    "fold_metrics": fold_table.astype(object).where(pd.notna(fold_table), None).to_dict(orient="records"),
    "paired_client_gain": {"mean": delta, "bootstrap_95_lower": float(lower), "bootstrap_95_upper": float(upper),
                           "full_queue_clients": len(full_clients), "resamples": 2000, "seed": 42},
    "fold_gains": fold_gain, "selected_method": selected_method,
    "selection_gate": "mean gain >= 0.05; lower bootstrap endpoint > 0; positive gains in >= 4/5 folds",
    "leak_control": {"clean_auc": float(clean_auc), "leaked_auc": float(leaked_auc), "leaked_input_removed": True},
    "oof_keys_sha256": u.fingerprint(oof_frame, u.KEYS), "oof_scores_sha256": u.fingerprint(oof_frame, ["oof_score", "fold"]),
    "limitations": ["one month only", "later coverage selects evaluation population", "client sizes uneven",
                    "model learned from other clients' later labels; not a deployment time test",
                    "earlier denominator can introduce regression-to-the-mean effects", "no observed editing benefit"]}
u.write_receipt(root, "w06_validation_metrics.json", receipt)
print("Committed receipts trace the source, split, scores, model choice, and limitations; OOF row data stay out of git.")

**Measured result:** OOF precision@20 is 57.17% for logistic regression and 38.26% for the rule. The paired-client gain interval is [+11.52, +26.74] points. My playbook method is **logistic_regression_oof**. This is a research choice for human review, not an automated editing system.

Committed receipts trace the source, split, scores, model choice, and limitations; OOF row data stay out of git.


## 5. Self-check

- [x] I named two paper findings, their pages, and a constructive methodology question for each.
- [x] I showed row-random versus client-held-out results and the shared-client counts.
- [x] I reran the fixed Week 5 method with broader grouped coverage and saved OOF receipts.
- [x] I audited timelines, target-derived columns, preprocessing, population selection, and model availability.
- [x] I showed real errors and repeated then removed one deliberate leak.
- [x] I rewrote claims around observed, measured, directional decision support.
- [x] The notebook ran top to bottom; its aggregate metrics JSON is included in the commit.
- [x] June, raw data, predictions, and credentials are excluded from git.